### Helpers

In [252]:
def disp_tweet(tweet):
    print(f"{tweet['user']['name']} (@{tweet['user']['screen_name']}): {tweet['text']}")

### Setup

In [253]:
from datasets import load_dataset

# Download the celebrity tweets dataset from Hugging Face
dataset_twitter = load_dataset("Jacobvs/CelebrityTweets")


In [254]:
# Load the dataset into a dataframe
all_data = dataset_twitter['train'].to_pandas()
print(f"Dataset loaded with {len(all_data)} entries, {len(all_data['username'].unique())} unique users.")
all_data.head()

Dataset loaded with 15886 entries, 90 unique users.


,index,date,id,username,text
0,0,2022-12-01 19:24:25,1598397624750866432,justinbieber,long time coming and excited to finally announ...
1,1,2022-09-07 01:14:34,1567320386546589696,justinbieber,https://t.co/qRY7ltRkV0
2,2,2022-07-07 19:33:51,1545128982143651840,justinbieber,1 year of preparation 🔥 @FreeFire_NA #GarenaFr...
3,3,2022-02-16 03:05:25,1493783548276314115,justinbieber,"Go for the Gold, Ladies!!!!!! Cannot wait to w..."
4,4,2021-12-30 22:33:21,1476682851324239873,justinbieber,The countdown begins… https://t.co/S5qZP92Ziz


In [255]:
# Filter for Elon Musk and Justin Bieber
USERS = ['elonmusk', 'justinbieber']
filtered_data = all_data[all_data['username'].isin(USERS)]
print(f"Filtered dataset contains {len(filtered_data)} entries from {len(filtered_data['username'].unique())} unique users.")

Filtered dataset contains 360 entries from 2 unique users.


### Cleaning

In [256]:
import re
cleaned_data = filtered_data.copy()

In [257]:
import emoji
import re

# Replace emojis with their shortcodes (e.g., 😀 becomes :grinning_face:)

def unemjoi(text):
    # Demojize and pad with spaces to ensure separation
    return emoji.demojize(text, delimiters=(" :", ": "))
cleaned_data['text'] = cleaned_data['text'].apply(unemjoi)

In [258]:
def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)

cleaned_data["text"] = cleaned_data["text"].apply(remove_punctuation)

In [259]:
def replace_urls(text):
    return re.sub(r'http\S+', '[URL]', text)

cleaned_data['text'] = cleaned_data['text'].apply(replace_urls)

In [260]:
cleaned_data['text'] = cleaned_data['text'].apply(str.lower)

In [261]:
def text_bold(text):
    return f"\033[1m{text}\033[0m"

def text_gray(text):
    return f"\033[90m{text}\033[0m"


In [262]:
def disp_tweet(user, text, max_line_length = 75, indentation = 0):
    
    USERNAME_INDENT = 1
    MIN_LINE_LENGTH = USERNAME_INDENT * 2 + len(user)  # Minimum line length to accommodate username and padding
    
    lines = text.split('\n')
    box_width = max(min(max(len(line) for line in lines), max_line_length), MIN_LINE_LENGTH) + 2  # Add padding
    
    def print_indented(line):
        print(" " * indentation + line)
        
    def print_line(line):
        print_indented("┃ " + text_gray(line.ljust(box_width - 2)) + " ┃")

    # Draw top of the box
    print_indented("┏" + "━" * USERNAME_INDENT + f' {text_bold(user)} ' + "━" * USERNAME_INDENT + "━" * (box_width - len(user) - USERNAME_INDENT * 2 - 2) + "┓")
    
    # Display the tweet text, max_line_length characters per line
    for line in lines:
        text_pos = 0
        while text_pos < len(line):
            print_line(
                line[text_pos:text_pos + max_line_length].ljust(box_width - 2)
            )
            text_pos += max_line_length
        
    # Draw bottom of the box
    print_indented("┗" + "━" * box_width + "┛")
    
for user in USERS:
    print(f"Tweets from {user}:")
    user_tweets = cleaned_data[cleaned_data['username'] == user].sample(5, random_state=42)  # Sample 5 tweets for display
    for _, tweet in user_tweets.iterrows():
        disp_tweet(user, tweet['text'], indentation=4)

Tweets from elonmusk:
    ┏━ elonmusk ━┓
    ┃ [url]      ┃
    ┗━━━━━━━━━━━━┛
    ┏━ elonmusk ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
    ┃ surround your house with treadmills set to jogging speed to stop walking de ┃
    ┃ ad ur welcome [url]                                                         ┃
    ┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
    ┏━ elonmusk ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
    ┃ and thats how we will land on mars [url] ┃
    ┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
    ┏━ elonmusk ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
    ┃ this test is at 50 throttle launch attempt next month will be at 90 [url] ┃
    ┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
    ┏━ elonmusk ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
    ┃ why did american media go from questioning the state and speaking truth to  ┃
    ┃ power to doing the

In [263]:
for user in USERS:
    print(f"\nSample tweets from {user}:")
    user_tweets = filtered_data[filtered_data['username'] == user]['text'].head(5)
    for tweet in user_tweets:
        print(f"- \"{tweet}\"")


Sample tweets from elonmusk:
- "The ability of Twitter advertising to reach the most influential people in the world is often not fully appreciated… https://t.co/gmW93tYzp1"
- "Twice as many people died in Japan last year as were born. Population freefall.

Rest of the world is trending to f… https://t.co/ewgJiFMmPa"
- "https://t.co/PHv5nGLZGe"
- "… https://t.co/a97BHpENpp"
- "You’re not gonna believe this, but we’re running a little late! 

Presentation starts in ~5 mins. https://t.co/Eck55BoLGj"

Sample tweets from justinbieber:
- "long time coming and excited to finally announce https://t.co/dFc8NBoFs0"
- "https://t.co/qRY7ltRkV0"
- "1 year of preparation 🔥 @FreeFire_NA #GarenaFreeFire #FF5thAnniversary #FFxJB https://t.co/U1YZpwJDrI https://t.co/mZ6pMfozXB"
- "Go for the Gold, Ladies!!!!!! Cannot wait to watch some of the best hockey ever!!! We are so proud of you! @HockeyCanada"
- "The countdown begins… https://t.co/S5qZP92Ziz"


In [264]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    filtered_data['text'],  # Use the 'text' column for features
    filtered_data['username'],  # Use the 'username' column for labels
    test_size=0.2,  # 20% of the data for testing
    random_state=42,  # For reproducibility
    stratify=filtered_data['username']  # Ensure the split is stratified by username
)
x_train

1203    An American group made false claims about Russ...
48                 📷: @rorykramer https://t.co/HMeQfAcfks
1258    Please comment on this matter https://t.co/Oip...
148     #HoldOn out now\nhttps://t.co/bjuAwdIQ5y https...
1107    Cybertruck at Tesla Engineering HQ https://t.c...
                              ...                        
1097                              https://t.co/UAWjEyhe5U
59      Join me on OCT 2nd! Tix on sale https://t.co/k...
1136                         True https://t.co/0IS89Mjlky
139     JUSTICE 3/19\nhttps://t.co/VSc6FCWSVs https://...
1109    High time I confessed I let the Doge out https...
Name: text, Length: 288, dtype: object

In [265]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
    max_features=50000,  # Limit to top 50000 features
    ngram_range=(1, 3),  # Use unigrams, bigrams, and trigrams
    min_df=5,  # Ignore terms that appear in less than 5 documents
)
x_train_tfidf = vectorizer.fit_transform(x_train)
x_test_tfidf = vectorizer.transform(x_test)

In [266]:
# Analyse the tfidf features
vectorizer_features = vectorizer.get_feature_names_out()
vectorizer_coefs = vectorizer.idf_
feature_importance = sorted(zip(vectorizer_features, vectorizer_coefs), key=lambda x: x[1], reverse=True)
print("Top 20 most important features:")
for feature, coef in feature_importance[:20]:
    print(f"  {(feature + ':').ljust(17)} {coef:.4f}")

Top 20 most important features:
  amazing:          4.8747
  anyone:           4.8747
  but:              4.8747
  can:              4.8747
  for the:          4.8747
  good:             4.8747
  media:            4.8747
  more:             4.8747
  not:              4.8747
  of the:           4.8747
  other:            4.8747
  team:             4.8747
  thekidlaroi:      4.8747
  tomorrow:         4.8747
  why:              4.8747
  at:               4.7205
  drewhouse:        4.7205
  justice https:    4.7205
  justice https co: 4.7205
  launch:           4.7205
